In [17]:
!pip uninstall -y torchao
!pip install -q torchao==0.16.0 transformers accelerate peft bitsandbytes pillow tqdm

Found existing installation: torchao 0.16.0
Uninstalling torchao-0.16.0:
  Successfully uninstalled torchao-0.16.0


In [18]:
import os
import json
import torch
import gc
import random

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import LoraConfig, get_peft_model, PeftModel
from torch.optim import AdamW

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os

BASE_DIR = "/content/drive/MyDrive/BanglaVision"
COCO_PATH = f"{BASE_DIR}/train2017"

if not os.path.exists(COCO_PATH):
    print("Downloading COCO train2017 to Drive...")

    !mkdir -p /content/tmp_coco
    %cd /content/tmp_coco

    !wget -c http://images.cocodataset.org/zips/train2017.zip
    !unzip -q train2017.zip

    !mv train2017 "{BASE_DIR}/"

    print("COCO saved to Drive")

else:
    print("COCO already exists skipping download")

/content/tmp_coco
--2026-05-03 12:43:33--  http://images.cocodataset.org/zips/train2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.183.173, 52.217.224.89, 52.216.49.225, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.183.173|:80... connected.
HTTP request sent, awaiting response... 206 Partial Content
Length: 19336861798 (18G), 19327188546 (18G) remaining [application/zip]
Saving to: ‘train2017.zip’

train2017.zip       100%[===================>]  18.01G  15.6MB/s    in 18m 9s  

2026-05-03 13:01:43 (16.9 MB/s) - ‘train2017.zip’ saved [19336861798/19336861798]

COCO saved to Drive


In [19]:
BASE_DIR = "/content/drive/MyDrive/BanglaVision"

IMAGE_ROOT = f"{BASE_DIR}/train2017"
JSON_PATH = f"{BASE_DIR}/coco_bn_final.json"

CHECKPOINT_DIR = f"{BASE_DIR}/coco_20k_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [20]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total COCO samples:", len(data))

random.seed(42)
data = random.sample(data, 20000)

print("Using samples:", len(data))

Total COCO samples: 591753
Using samples: 20000


In [21]:
for item in data:
    if not item["image"].startswith("/"):
        item["image"] = os.path.join(IMAGE_ROOT, item["image"])

In [22]:
class CocoDataset(Dataset):
    def __init__(self, data, processor):
        self.data = data
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        try:
            image = Image.open(item["image"]).convert("RGB")
        except:
            image = Image.new("RGB", (224, 224))

        text = "এই ছবিটি বর্ণনা কর: " + item["caption_bn"]

        encoding = self.processor(
            images=image,
            text=text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=80
        )

        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding["labels"] = encoding["input_ids"].clone()

        return encoding

In [23]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = True

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

In [24]:
def disable_all_checkpointing(model):
    for module in model.modules():
        if hasattr(module, "gradient_checkpointing"):
            module.gradient_checkpointing = False

disable_all_checkpointing(model)

In [25]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,621,440 || all params: 3,747,383,296 || trainable%: 0.0700


In [26]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,621,440 || all params: 3,747,383,296 || trainable%: 0.0700


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [27]:
def extract_step(x):
    return int(x.split("_")[-1])

valid = []

for ckpt in os.listdir(CHECKPOINT_DIR):
    path = os.path.join(CHECKPOINT_DIR, ckpt)

    if os.path.exists(os.path.join(path, "adapter_config.json")):
        valid.append(ckpt)

valid = sorted(valid, key=extract_step)

if len(valid) > 0:
    last = os.path.join(CHECKPOINT_DIR, valid[-1])
    print("Resuming from:", last)
    model = PeftModel.from_pretrained(model, last)
else:
    print("Starting fresh")

Starting fresh


In [28]:
STEP_SIZE = 5000
splits = split_data(data, STEP_SIZE)

print("Total chunks:", len(splits))

Total chunks: 4


In [29]:
BATCH_SIZE = 1
EPOCHS = 2

for epoch in range(EPOCHS):

    print(f"\nEPOCH {epoch+1}")

    for i, split in enumerate(splits):

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"coco20k_step_{epoch}_{(i+1)*STEP_SIZE}"
        )

        if os.path.exists(os.path.join(checkpoint_path, "adapter_model.safetensors")):
            print(f"Skipping chunk {i+1}")
            continue

        print(f"\nTraining chunk {i+1}/{len(splits)}")

        dataset = CocoDataset(split, processor)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

        optimizer = AdamW(model.parameters(), lr=2e-5)

        model.train()
        pbar = tqdm(loader)

        for batch in pbar:
            try:
                batch = {k: v.to("cuda") for k, v in batch.items()}

                outputs = model(**batch)
                loss = outputs.loss

                if torch.isnan(loss):
                    print("Skipping NaN")
                    continue

                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                optimizer.step()
                optimizer.zero_grad()

                pbar.set_description(f"Loss: {loss.item():.4f}")

            except Exception as e:
                print("Skipping batch:", e)
                continue

        model.save_pretrained(checkpoint_path)
        processor.save_pretrained(checkpoint_path)

        print(f"Saved: {checkpoint_path}")

        torch.cuda.empty_cache()
        gc.collect()


EPOCH 1

Training chunk 1/4


Loss: 0.2937: 100%|██████████| 5000/5000 [13:53<00:00,  6.00it/s]


Saved: /content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_5000

Training chunk 2/4


Loss: 0.1276: 100%|██████████| 5000/5000 [30:33<00:00,  2.73it/s]


Saved: /content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_10000

Training chunk 3/4


...

Saved: /content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_1_10000

Training chunk 3/4


Loss: 0.1962: 100%|██████████| 5000/5000 [13:22<00:00,  6.23it/s]


Saved: /content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_1_15000

Training chunk 4/4


Loss: 0.0525: 100%|██████████| 5000/5000 [13:24<00:00,  6.21it/s]


Saved: /content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_1_20000


In [32]:
import os

print("Checking checkpoint dir...")

path = "/content/drive/MyDrive/BanglaVision"

for root, dirs, files in os.walk(path):
    if "coco20k" in root:
        print(root)
        print(files)

Checking checkpoint dir...
/content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_5000
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']
/content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_10000
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']
/content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_15000
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']
/content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_20000
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']
/content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_1_5000
['README.md', 'adapter_model.safetensors', 'adapt

In [37]:
!zip -r coco_20k_checkpoints.zip /content/drive/MyDrive/BanglaVision/coco_20k_checkpoints

  adding: content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/ (stored 0%)
  adding: content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_5000/ (stored 0%)
  adding: content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_5000/README.md (deflated 66%)
  adding: content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_5000/adapter_model.safetensors (deflated 8%)
  adding: content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_5000/adapter_config.json (deflated 59%)
  adding: content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_5000/tokenizer_config.json (deflated 46%)
  adding: content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_5000/tokenizer.json (deflated 82%)
  adding: content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_5000/processor_config.json (deflated 50%)
  adding: content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_0_10000/ (stored 0%)
  

In [38]:
from google.colab import files
files.download("coco_20k_checkpoints.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>